# Session 11: Building a Vector Store

**Notebook 3 of 3: Hands-on** · about 45 minutes

A vector store keeps vectors with their text and metadata, and returns the ones most similar to a query vector. Products such as Chroma, Qdrant, Pinecone or PostgreSQL's pgvector do this at scale, with indexes, persistence and filtering. Underneath, the idea is small enough to build in an afternoon, and building it is the best way to understand what those products do for you.

By the end of this notebook you will be able to:

1. **Implement** a vector store with NumPy: add, search with a minimum score, filter by metadata.
2. **Persist** it to disk with the embedding model's ID, and **refuse** to mix models.
3. **Explain** why brute-force search is fine for thousands of vectors, and what changes at millions.
4. **Use** Chroma with your own embeddings, and **compare** its results with yours.

The workbook's `VectorStore` follows this design.

> **Kernel:** the course environment. Offline: it uses the recorded embeddings from `embeddings.py`, like notebook 1.

## Setup: the handbook's chunks and vectors

The same loading and chunking code as notebook 1, condensed, so that the chunks, and therefore their recorded vectors, are identical.

In [ ]:
import json
import re
import tempfile
import time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from pypdf import PdfReader

from embeddings import Embedder

embedder = Embedder(
    live=False
)  # True: compute vectors for new questions (free; downloads ~67 MB once)
HANDBOOK = Path("..") / "data" / "handbook"


@dataclass(frozen=True)
class Document:
    source: str
    title: str
    text: str
    page: int | None


@dataclass(frozen=True)
class Chunk:
    text: str
    source: str
    title: str
    section: str | None
    page: int | None
    index: int

    @property
    def embedding_text(self) -> str:
        heading = " / ".join(part for part in (self.title, self.section) if part)
        return f"{heading}\n{self.text}"


def load_folder(folder: Path) -> list[Document]:
    documents = []
    for path in sorted(folder.iterdir()):
        if path.suffix == ".md":
            text = path.read_text(encoding="utf-8")
            first = text.splitlines()[0] if text else ""
            title = first.lstrip("# ").strip() if first.startswith("# ") else path.stem
            documents.append(Document(path.name, title, text, None))
        elif path.suffix == ".pdf":
            reader = PdfReader(path)
            title = (reader.metadata.title if reader.metadata else None) or path.stem
            for number, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                if text.strip():
                    documents.append(Document(path.name, title, text, number))
    return documents


def split_sections(text: str) -> list[tuple[str | None, str]]:
    sections, heading, lines = [], None, []
    for line in text.splitlines():
        if line.startswith("## "):
            sections.append((heading, "\n".join(lines)))
            heading, lines = line[3:].strip(), []
        elif not line.startswith("# "):
            lines.append(line)
    sections.append((heading, "\n".join(lines)))
    return [(h, body.strip()) for h, body in sections if body.strip()]


def split_text(text: str, size: int, overlap: int) -> list[str]:
    units = re.split(r"(?<=[.!?:])\s+|\n+", " ".join(text.split()) if "\n\n" not in text else text)
    units = [" ".join(u.split()) for u in units if u.strip()]
    pieces, current = [], []
    for unit in units:
        if current and len(" ".join(current + [unit])) > size:
            pieces.append(" ".join(current))
            carried = []
            for previous in reversed(current):
                if len(" ".join([previous] + carried)) > overlap:
                    break
                carried.insert(0, previous)
            current = carried
        current.append(unit)
    if current:
        pieces.append(" ".join(current))
    return pieces


def chunk_documents(documents: list[Document], size: int = 500, overlap: int = 100) -> list[Chunk]:
    chunks, counters = [], Counter()
    for document in documents:
        sections = (
            split_sections(document.text) if document.page is None else [(None, document.text)]
        )
        for section, body in sections:
            for piece in split_text(body, size, overlap):
                chunks.append(
                    Chunk(
                        piece,
                        document.source,
                        document.title,
                        section,
                        document.page,
                        counters[document.source],
                    )
                )
                counters[document.source] += 1
    return chunks


chunks = chunk_documents(load_folder(HANDBOOK))
vectors = embedder.embed([c.embedding_text for c in chunks])
print(f"{len(chunks)} chunks, vectors {vectors.shape}, model {embedder.model_id}")

## Part 1: The core: add and search

A vector store is three things kept in the same order: a matrix of vectors (one row per chunk), the chunks themselves, and the ID of the model that made the vectors.

Search is one matrix product. If every row has length 1, the dot product of a row with a unit-length query **is** the cosine similarity, so the store normalises vectors when they're added. Then it doesn't depend on whether the embedding model already does.

In [ ]:
def normalise(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=-1, keepdims=True)
    return matrix / np.where(norms == 0, 1, norms)  # leave all-zero rows as they are


@dataclass(frozen=True)
class Hit:
    score: float
    chunk: Chunk


class VectorStore:
    def __init__(self, model_id: str, dimensions: int) -> None:
        self.model_id = model_id
        self.vectors = np.empty((0, dimensions), dtype=np.float32)
        self.chunks: list[Chunk] = []

    def __len__(self) -> int:
        return len(self.chunks)

    def add(self, chunks: list[Chunk], vectors: np.ndarray) -> None:
        if len(chunks) != len(vectors):
            raise ValueError(f"{len(chunks)} chunks but {len(vectors)} vectors")
        if vectors.shape[1] != self.vectors.shape[1]:
            raise ValueError(
                f"vectors have {vectors.shape[1]} dimensions; this store holds {self.vectors.shape[1]}"
            )
        self.vectors = np.vstack([self.vectors, normalise(vectors.astype(np.float32))])
        self.chunks.extend(chunks)

    def search(self, query: np.ndarray, k: int = 3, min_score: float = -1.0) -> list[Hit]:
        if not self.chunks:
            return []
        scores = self.vectors @ normalise(query.astype(np.float32))
        best = np.argsort(scores)[::-1][:k]
        return [Hit(float(scores[i]), self.chunks[i]) for i in best if scores[i] >= min_score]


store = VectorStore(embedder.model_id, vectors.shape[1])
store.add(chunks, vectors)
question = "The Berlin office VPN keeps disconnecting."
for hit in store.search(embedder.embed([question])[0]):
    print(f"{hit.score:.3f}  {hit.chunk.source} | {hit.chunk.section}")

**Predict:** the store refuses a batch of vectors with the wrong number of dimensions. What would happen without that check, if someone added 1,536-dimensional OpenAI vectors to this 384-dimensional store?

<details>
<summary>Show answer</summary>

`np.vstack` would fail with a shape error, which is at least loud. The dangerous case is two models with the **same** number of dimensions: the vectors stack happily, and search returns meaningless scores, with no error anywhere. That's why the store also records the model ID, and Part 3 checks it when loading.

</details>

**Modify:** search for `"How do I fail over the orders database?"` with `k=5`. Then add `min_score=0.7`. How many hits survive? (A new question needs live embeddings: create the embedder with `live=True` in the first code cell, and run the notebook again.)

## Part 2: Does brute force scale?

Every search scores **every** vector. That sounds slow. Measure it, with random vectors standing in for a much bigger corpus:

In [ ]:
rng = np.random.default_rng(seed=11)
query = normalise(rng.normal(size=384).astype(np.float32))

for count in (1_000, 100_000, 1_000_000):
    matrix = normalise(rng.normal(size=(count, 384)).astype(np.float32))
    started = time.perf_counter()
    scores = matrix @ query
    top = np.argpartition(scores, -5)[-5:]  # the 5 best, without sorting everything
    top = top[np.argsort(scores[top])[::-1]]
    elapsed = (time.perf_counter() - started) * 1000
    print(
        f"{count:>9,} vectors: {elapsed:7.1f} ms per search, {matrix.nbytes / 1e6:6.0f} MB of vectors"
    )
    del matrix

**Explain:** NumPy's matrix product runs in optimised native code, so a million 384-dimensional vectors take tens of milliseconds per search on a laptop (your timings will differ), and 1.5 GB of memory. `argpartition` finds the top k without sorting all the scores.

| Corpus | Brute force with NumPy | What changes |
|---|---|---|
| Thousands of chunks (a handbook, a product's docs) | Milliseconds; trivial memory | Nothing: a NumPy store is fine |
| Hundreds of thousands | Tens of milliseconds; hundreds of MB | Fine for one process; persistence and updates start to matter |
| Millions and more, many users | Memory and latency add up | **Approximate nearest-neighbour** (ANN) indexes such as HNSW: much faster, slightly approximate. That's what vector databases provide |

The capstone's corpus is in the first row. Choose a vector database for the features you need (persistence, filtering, concurrent updates, scale), not because "RAG needs a vector database".

## Part 3: Persistence, and refusing to mix models

Embedding is the expensive step of ingestion, so the store is saved to disk and loaded at query time. Two files: the vectors as a NumPy array, and everything else as JSON, including the model ID.

In [ ]:
class PersistentVectorStore(VectorStore):
    def save(self, folder: Path) -> None:
        folder.mkdir(parents=True, exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        meta = {
            "model_id": self.model_id,
            "dimensions": self.vectors.shape[1],
            "chunks": [asdict(chunk) for chunk in self.chunks],
        }
        (folder / "store.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")

    @classmethod
    def load(cls, folder: Path, *, expected_model_id: str) -> "PersistentVectorStore":
        meta = json.loads((folder / "store.json").read_text(encoding="utf-8"))
        if meta["model_id"] != expected_model_id:
            raise ValueError(
                f"the store was built with {meta['model_id']!r}, but queries use {expected_model_id!r}: "
                "re-ingest the documents"
            )
        store = cls(meta["model_id"], meta["dimensions"])
        store.vectors = np.load(folder / "vectors.npy")
        store.chunks = [Chunk(**fields) for fields in meta["chunks"]]
        if len(store.vectors) != len(store.chunks):
            raise ValueError("vectors.npy and store.json don't match: re-ingest the documents")
        return store


with tempfile.TemporaryDirectory() as workdir:
    folder = Path(workdir) / "index"
    original = PersistentVectorStore(embedder.model_id, vectors.shape[1])
    original.add(chunks, vectors)
    original.save(folder)
    print(
        "saved:",
        sorted(p.name for p in folder.iterdir()),
        f"{(folder / 'vectors.npy').stat().st_size / 1024:.0f} KB",
    )

    reloaded = PersistentVectorStore.load(folder, expected_model_id=embedder.model_id)
    query = embedder.embed([question])[0]
    print(
        "same results after reload:",
        [h.chunk.index for h in reloaded.search(query)]
        == [h.chunk.index for h in original.search(query)],
    )

    try:
        PersistentVectorStore.load(folder, expected_model_id="text-embedding-3-small")
    except ValueError as error:
        print("refused:", error)

**Engineering Insight:** keep the store's format boring and inspectable. `store.json` is readable in any editor, so you can check what was indexed and from which file. Never use `pickle` for files you might load from elsewhere: unpickling can run arbitrary code. `np.load` refuses pickled objects by default (`allow_pickle=False`) for the same reason.

## Part 4: Filtering by metadata

"Search only the runbooks" or "only documents owned by the network team" are filters on metadata. The simple, correct approach: decide which chunks are eligible first, then rank only those.

In [ ]:
def search_where(store: VectorStore, query: np.ndarray, k: int = 3, **conditions) -> list[Hit]:
    eligible = [
        i
        for i, chunk in enumerate(store.chunks)
        if all(getattr(chunk, field) == value for field, value in conditions.items())
    ]
    if not eligible:
        return []
    scores = store.vectors[eligible] @ normalise(query.astype(np.float32))
    best = np.argsort(scores)[::-1][:k]
    return [Hit(float(scores[j]), store.chunks[eligible[j]]) for j in best]


query = embedder.embed(["How often must the status page be updated during a P1?"])[0]
print("everything:  ", [(round(h.score, 2), h.chunk.source) for h in store.search(query)])
print(
    "only the PDF:",
    [
        (round(h.score, 2), h.chunk.source, h.chunk.page)
        for h in search_where(store, query, source="policy-incident-communication.pdf")
    ],
)

Filtering **before** ranking matters. Ranking first, taking the top k, then filtering can return fewer than k results, or none, even when eligible chunks exist further down the list. Vector databases document which of the two they do; check before relying on filters.

## Part 5: The same thing with Chroma

[Chroma](https://docs.trychroma.com/) is an open-source vector database that runs inside your Python process, with optional persistence to disk. You can give it your own vectors, so the same embeddings serve both stores and the results are directly comparable:

In [ ]:
import chromadb
from chromadb.config import Settings

client = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False))  # in memory
collection = client.create_collection(
    name="handbook",
    embedding_function=None,  # we supply the vectors ourselves
    configuration={"hnsw": {"space": "cosine"}},  # rank by cosine distance
)


def chroma_metadata(chunk: Chunk) -> dict:
    """Chroma metadata values must be str, int, float or bool: drop the Nones."""
    return {
        key: value for key, value in asdict(chunk).items() if value is not None and key != "text"
    }


collection.add(
    ids=[f"{c.source}#{c.index}" for c in chunks],
    embeddings=vectors.tolist(),
    documents=[c.text for c in chunks],
    metadatas=[chroma_metadata(c) for c in chunks],
)

query = embedder.embed([question])[0]
result = collection.query(query_embeddings=[query.tolist()], n_results=3)
print("Chroma:", [(round(1 - d, 3), i) for d, i in zip(result["distances"][0], result["ids"][0])])
print(
    "NumPy: ",
    [(round(h.score, 3), f"{h.chunk.source}#{h.chunk.index}") for h in store.search(query)],
)

filtered = collection.query(
    query_embeddings=[query.tolist()], n_results=2, where={"source": "runbook-vpn-outage.md"}
)
print("Chroma with a filter:", filtered["ids"][0])

**Explain:**

- Chroma reports **distances**: with the cosine space, distance = 1 − similarity. Converted, the scores and the ranking match the NumPy store.
- Chroma uses an **HNSW** index, an approximate nearest-neighbour structure. With a few dozen vectors it finds exactly the same results; on large collections it can occasionally miss a true neighbour, in exchange for much faster search.
- `embedding_function=None` matters. Without it, Chroma would embed documents itself with its own default model, which it downloads on first use. Then you'd have **two** embedding models: Chroma's for the documents, and yours for the queries.
- `anonymized_telemetry=False` switches off Chroma's usage reporting. Check what any library sends home, and decide deliberately.
- Metadata can't contain `None`, so it's filtered out first. Small format rules like this are where integrations break.
- `chromadb.PersistentClient(path=...)` stores the collection on disk instead of in memory. On Windows it keeps its files open while the client exists, so give each run its own folder in tests.

### Which one?

| | NumPy store | Chroma |
|---|---|---|
| Code you own | About 60 lines | A few calls |
| Dependencies | NumPy | A substantial package, with dozens of dependencies of its own |
| Search | Exact | Approximate (HNSW), exact at small sizes |
| Persistence | Your format (`.npy` + JSON) | Built in (SQLite plus index files) |
| Filtering | Your code | `where=` on metadata, and on document text |
| Updates and deletes | Your code | Built in, by id |
| Good for | Learning; small, mostly static corpora; tests | Growing corpora; filtering; less code to maintain |

The workbook builds the NumPy store, so you know exactly what a vector store does. The capstone uses Chroma, so you know what a library adds.

## Key takeaways

- A vector store is vectors, chunks and a model ID, in the same order. Search is a matrix product and a top-k selection.
- Normalise vectors when you store them, and the dot product is the cosine similarity.
- Brute force is fast enough for thousands to hundreds of thousands of vectors; vector databases add approximate indexes, persistence, filtering and updates.
- Save the model ID with the vectors, and refuse to load a store built with a different model.
- Filter before ranking, or filters can return fewer results than exist.
- With Chroma, supply your own embeddings, choose cosine distance, and switch off telemetry.

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then start the workbook.